# RetinaGuard — Deep Learning Diabetic Retinopathy Classification
### CECS 551 | Phase 4 | Spring 2026 | California State University, Long Beach

| | |
|---|---|
| **Team** | Dhathresh Prathap Kora · Vinay Krishna · Jaswanth Maddineni · Rohan Sanda |
| **Dataset** | [Diabetic Retinopathy 224×224 Gaussian-Filtered](https://www.kaggle.com/datasets/sovitrath/diabetic-retinopathy-224x224-gaussian-filtered) |
| **Task** | 5-class DR severity classification (No DR → Proliferative) |
| **Metric** | Quadratic Weighted Kappa (QWK) + AUROC for referable DR |

---

---
## Section 1 — Setup & Configuration

In [1]:
import subprocess, sys, time

def pip_install(pkg, fallbacks=None):
    """Install pkg with fallbacks. Silent on success, warns on failure."""
    candidates = [pkg] + (fallbacks or [])
    for name in candidates:
        try:
            subprocess.check_call(
                [sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', name],
                stderr=subprocess.DEVNULL,
                timeout=120   # 2 min max per package
            )
            print(f'  ✅ {name}')
            return name
        except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
            print(f'  ⚠️  {name} failed, trying next...')
    raise RuntimeError(f'Could not install any of: {candidates}')

t0 = time.time()
pip_install('timm')
pip_install('grad-cam', fallbacks=['pytorch-grad-cam', 'gradcam'])
pip_install('scikit-learn')
pip_install('seaborn')
pip_install('opencv-python-headless', fallbacks=['opencv-python'])
print(f'\n✅ All packages installed in {time.time()-t0:.0f}s')

  ✅ timm
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 148.5 MB/s eta 0:00:00
  ✅ grad-cam
  ✅ scikit-learn
  ✅ seaborn
  ✅ opencv-python-headless

✅ All packages installed in 24s


In [2]:
import os, random, time, copy, warnings, textwrap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # prevents browser OOM — do NOT change
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import GradScaler
import torch.cuda.amp as amp
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, average_precision_score, precision_recall_curve,
    roc_curve, cohen_kappa_score, f1_score, precision_score, recall_score
)
from sklearn.preprocessing import label_binarize

try:
    from pytorch_grad_cam import GradCAMPlusPlus
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    print('✅ Loaded: pytorch_grad_cam')
except ImportError:
    from gradcam import GradCAMPlusPlus
    from gradcam.utils.image import show_cam_on_image
    from gradcam.utils.model_targets import ClassifierOutputTarget
    print('✅ Loaded: gradcam')

warnings.filterwarnings('ignore')

# ── Global plot style
plt.rcParams.update({
    'figure.dpi'       : 90,
    'font.size'        : 10,
    'axes.titlesize'   : 11,
    'axes.labelsize'   : 10,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'font.family'      : 'DejaVu Sans',
})
sns.set_palette('husl')
print('✅ Imports done.')

✅ Loaded: pytorch_grad_cam
✅ Imports done.


In [3]:
# ── Reproducibility
SEED = 42

def set_seed(s=SEED):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed()
print(f'🌱 Global seed set to {SEED} — fully reproducible run.')

🌱 Global seed set to 42 — fully reproducible run.


In [4]:
CFG = {
    # ── Paths
    'dataset_root': '/kaggle/input/datasets/sovitrath/diabetic-retinopathy-224x224-gaussian-filtered/gaussian_filtered_images/gaussian_filtered_images',
    'csv_path'    : '/kaggle/input/datasets/sovitrath/diabetic-retinopathy-224x224-gaussian-filtered/train.csv',
    'output_dir'  : '/kaggle/working/outputs',

    # ── Data
    'img_size'    : 224,
    'num_classes' : 5,
    'class_names' : ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative'],
    'class_dirs'  : {0:'No_DR', 1:'Mild', 2:'Moderate', 3:'Severe', 4:'Proliferate_DR'},

    # ── Training
    'batch_size'  : 32,
    'num_epochs'  : 20,
    'lr'          : 1e-4,
    'weight_decay': 1e-4,
    'patience'    : 5,
    'use_amp'     : True,
    'use_clahe'   : True,

    # ── Splits
    'train_frac'  : 0.70,
    'val_frac'    : 0.15,
    'test_frac'   : 0.15,

    # ── Loss
    'focal_gamma' : 2.0,
    'focal_alpha' : 0.25,   # NEW: scale focal so magnitudes match CE
    'label_smooth': 0.05,   # reduced from 0.1 — less smoothing = better minority recall

    # ── Eval
    'tta_n'       : 5,

    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

# ── Output directory tree
for d in ['figures', 'tables', 'checkpoints', 'gradcam']:
    Path(f"{CFG['output_dir']}/{d}").mkdir(parents=True, exist_ok=True)

FIG  = f"{CFG['output_dir']}/figures"
TAB  = f"{CFG['output_dir']}/tables"
CKPT = f"{CFG['output_dir']}/checkpoints"
GCAM = f"{CFG['output_dir']}/gradcam"

# ── Palette (consistent across all figures)
COLORS = ['#27ae60','#f39c12','#e67e22','#e74c3c','#8e44ad']
LABELS = CFG['class_names']

print('━'*55)
print('  RetinaGuard — Phase 4 Configuration')
print('━'*55)
for k, v in CFG.items():
    print(f'  {k:<20s}: {v}')
print('━'*55)
print(f'  Device : {CFG["device"]}')
if torch.cuda.is_available():
    print(f'  GPU    : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  RetinaGuard — Phase 4 Configuration
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  dataset_root        : /kaggle/input/datasets/sovitrath/diabetic-retinopathy-224x224-gaussian-filtered/gaussian_filtered_images/gaussian_filtered_images
  csv_path            : /kaggle/input/datasets/sovitrath/diabetic-retinopathy-224x224-gaussian-filtered/train.csv
  output_dir          : /kaggle/working/outputs
  img_size            : 224
  num_classes         : 5
  class_names         : ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
  class_dirs          : {0: 'No_DR', 1: 'Mild', 2: 'Moderate', 3: 'Severe', 4: 'Proliferate_DR'}
  batch_size          : 32
  num_epochs          : 20
  lr                  : 0.0001
  weight_decay        : 0.0001
  patience            : 5
  use_amp             : True
  use_clahe           : True
  train_frac          : 0.7
  val_frac            : 0.15
  test_frac           : 0.15
  focal_gamma   

---
## Section 2 — Data Loading & Validation

In [5]:
df_raw = pd.read_csv(CFG['csv_path'])
print(f'CSV shape  : {df_raw.shape}')
print(f'Columns    : {df_raw.columns.tolist()}')
print(df_raw.head())

CSV shape  : (3662, 2)
Columns    : ['id_code', 'diagnosis']
        id_code  diagnosis
0  000c1434d8d7          2
1  001639a390f0          4
2  0024cdab0c1e          1
3  002c21358ce6          0
4  005b95c28852          0


In [6]:
def build_path(row):
    label  = int(row['diagnosis'])
    folder = CFG['class_dirs'][label]
    img_id = str(row['id_code']).strip()
    for ext in ['.png', '.jpg', '.jpeg', '']:
        p = Path(CFG['dataset_root']) / folder / f'{img_id}{ext}'
        if p.exists():
            return str(p)
    return None

df_raw['filepath'] = df_raw.apply(build_path, axis=1)
df_raw['label']    = df_raw['diagnosis'].astype(int)

n_missing = df_raw['filepath'].isna().sum()
df = df_raw.dropna(subset=['filepath']).reset_index(drop=True)

print(f'Total rows   : {len(df_raw)}')
print(f'Missing files: {n_missing}')
print(f'Valid images : {len(df)}')
print()
print(df[['id_code','label','filepath']].head())

# Quick file integrity check
print(f'\n✅ All {len(df)} image paths verified on disk.')

Total rows   : 3662
Missing files: 0
Valid images : 3662

        id_code  label                                           filepath
0  000c1434d8d7      2  /kaggle/input/datasets/sovitrath/diabetic-reti...
1  001639a390f0      4  /kaggle/input/datasets/sovitrath/diabetic-reti...
2  0024cdab0c1e      1  /kaggle/input/datasets/sovitrath/diabetic-reti...
3  002c21358ce6      0  /kaggle/input/datasets/sovitrath/diabetic-reti...
4  005b95c28852      0  /kaggle/input/datasets/sovitrath/diabetic-reti...

✅ All 3662 image paths verified on disk.


---
## Section 3 — Exploratory Data Analysis (EDA)

In [7]:
# ════════════════════════════════════════════
# Fig 1: Class Distribution — bar + pie + imbalance annotation
# ════════════════════════════════════════════
counts = df['label'].value_counts().sort_index()

fig = plt.figure(figsize=(15, 5))
fig.suptitle('Figure 1 — DR Severity Class Distribution', fontsize=14, fontweight='bold', y=1.01)
gs  = gridspec.GridSpec(1, 3, width_ratios=[2, 1.2, 1], figure=fig)

# ── Bar chart
ax0 = fig.add_subplot(gs[0])
bar_colors = COLORS
bars = ax0.bar(LABELS, counts.values, color=bar_colors, edgecolor='white', linewidth=1.2, width=0.6)
ax0.axhline(counts.mean(), ls='--', color='#555', lw=1.5, label=f'Mean = {int(counts.mean())}')
ax0.set_title('Image Count per Class', fontweight='bold')
ax0.set_ylabel('Number of Images')
ax0.set_xlabel('DR Severity Class')
ax0.legend()
for bar, val in zip(bars, counts.values):
    ax0.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 18,
             f'{val:,}', ha='center', fontweight='bold', fontsize=9)

# ── Pie chart
ax1 = fig.add_subplot(gs[1])
wedges, texts, autotexts = ax1.pie(
    counts.values, colors=COLORS, startangle=140,
    autopct='%1.1f%%', pctdistance=0.75,
    wedgeprops=dict(linewidth=1.5, edgecolor='white')
)
for at in autotexts:
    at.set_fontsize(8); at.set_fontweight('bold')
ax1.set_title('Class Proportion', fontweight='bold')
ax1.legend(LABELS, loc='lower center', bbox_to_anchor=(0.5,-0.15), fontsize=7, ncol=3)

# ── Imbalance stats table
ax2 = fig.add_subplot(gs[2])
ax2.axis('off')
table_data = [['Class','Count','%']] + [
    [LABELS[i], f'{counts[i]:,}', f'{counts[i]/counts.sum()*100:.1f}%']
    for i in range(5)
] + [['Imbalance', f'{counts.max()/counts.min():.1f}×', '(max/min)']]
tbl = ax2.table(cellText=table_data[1:], colLabels=table_data[0],
                cellLoc='center', loc='center', bbox=[0,0,1,1])
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
for (r,c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2c3e50'); cell.set_text_props(color='white', fontweight='bold')
    elif r == 6:
        cell.set_facecolor('#fadbd8')
    elif r % 2 == 0:
        cell.set_facecolor('#f8f9fa')
ax2.set_title('Distribution Table', fontweight='bold')

plt.tight_layout()
sp = f'{FIG}/fig1_class_distribution.png'
plt.savefig(sp, bbox_inches='tight', dpi=120)
plt.show(); plt.close('all')
print(f'Saved: {sp}')

dist_df = pd.DataFrame({'Class':range(5),'Label':LABELS,
                        'Count':counts.values,
                        'Pct':(counts.values/counts.sum()*100).round(1)})
dist_df.to_csv(f'{TAB}/class_distribution.csv', index=False)
print(f'Imbalance ratio: {counts.max()/counts.min():.1f}×')

Saved: /kaggle/working/outputs/figures/fig1_class_distribution.png
Imbalance ratio: 9.4×


In [8]:
# ════════════════════════════════════════════
# Fig 2: Sample Images Grid — 5 classes × 4 samples with class labels
# ════════════════════════════════════════════
fig, axes = plt.subplots(5, 4, figsize=(13, 16))
fig.suptitle('Figure 2 — Sample Retinal Fundus Images per DR Severity Class',
             fontsize=13, fontweight='bold')

for row_i, cls in enumerate(range(5)):
    samples = df[df['label']==cls].sample(min(4, len(df[df['label']==cls])), random_state=SEED)
    for col_i, (_, srow) in enumerate(samples.iterrows()):
        img = cv2.imread(srow['filepath'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[row_i][col_i].imshow(img)
        axes[row_i][col_i].axis('off')
        if col_i == 0:
            axes[row_i][col_i].set_ylabel(
                f'Grade {cls}\n{LABELS[cls]}',
                fontsize=9, fontweight='bold', rotation=0,
                labelpad=60, va='center'
            )
        axes[row_i][col_i].set_title(f'#{col_i+1}', fontsize=7, pad=2)
    # Color border per class
    for col_i in range(4):
        for spine in axes[row_i][col_i].spines.values():
            spine.set_visible(True)
            spine.set_edgecolor(COLORS[cls])
            spine.set_linewidth(2)

plt.tight_layout()
sp = f'{FIG}/fig2_sample_images.png'
plt.savefig(sp, bbox_inches='tight', dpi=100)
plt.show(); plt.close('all')
print(f'Saved: {sp}')

Saved: /kaggle/working/outputs/figures/fig2_sample_images.png


In [9]:
# ════════════════════════════════════════════
# Fig 3: Pixel Intensity Distributions — KDE overlay
# ════════════════════════════════════════════
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
fig.suptitle('Figure 3 — Pixel Intensity Distribution per DR Class', fontsize=12, fontweight='bold')

intensity_stats = []
for cls in range(5):
    samples = df[df['label']==cls].sample(min(40, len(df[df['label']==cls])), random_state=SEED)
    all_pixels, means, stds = [], [], []
    for _, row in samples.iterrows():
        img = cv2.imread(row['filepath'], cv2.IMREAD_GRAYSCALE)
        if img is not None:
            all_pixels.extend(img.flatten().tolist())
            means.append(img.mean()); stds.append(img.std())
    mu, sd = np.mean(means), np.mean(stds)
    axes[cls].hist(all_pixels, bins=60, color=COLORS[cls], alpha=0.65, density=True, label='Density')
    axes[cls].axvline(mu, color='black', ls='--', lw=1.5, label=f'μ={mu:.1f}')
    axes[cls].axvspan(mu-sd, mu+sd, alpha=0.1, color=COLORS[cls], label=f'±1σ')
    axes[cls].set_title(f'Grade {cls}: {LABELS[cls]}', fontweight='bold', fontsize=9)
    axes[cls].set_xlabel('Pixel Intensity (0–255)', fontsize=8)
    if cls == 0: axes[cls].set_ylabel('Density', fontsize=8)
    axes[cls].legend(fontsize=6.5)
    axes[cls].set_xlim(0, 255)
    intensity_stats.append({'Class':cls,'Label':LABELS[cls],'Mean':round(mu,2),'Std':round(sd,2)})

plt.tight_layout()
sp = f'{FIG}/fig3_pixel_intensity.png'
plt.savefig(sp, bbox_inches='tight', dpi=110)
plt.show(); plt.close('all')
print(f'Saved: {sp}')

stats_df = pd.DataFrame(intensity_stats)
stats_df.to_csv(f'{TAB}/intensity_stats.csv', index=False)
print(stats_df.to_string(index=False))

Saved: /kaggle/working/outputs/figures/fig3_pixel_intensity.png
 Class         Label   Mean   Std
     0         No DR 135.80 22.14
     1          Mild 135.97 15.51
     2      Moderate 135.93 15.61
     3        Severe 135.93 15.98
     4 Proliferative 136.17 16.05


In [10]:
# ════════════════════════════════════════════
# Fig 4: Brightness & Contrast — styled boxplots with stripplot overlay
# ════════════════════════════════════════════
brightness_data, contrast_data = {}, {}
for cls in range(5):
    samples = df[df['label']==cls].sample(min(80, len(df[df['label']==cls])), random_state=SEED)
    b, c = [], []
    for _, row in samples.iterrows():
        img = cv2.imread(row['filepath'], cv2.IMREAD_GRAYSCALE)
        if img is not None:
            b.append(img.mean()); c.append(img.std())
    brightness_data[LABELS[cls]] = b
    contrast_data[LABELS[cls]]   = c

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Figure 4 — Image Quality: Brightness & Contrast per Class', fontsize=12, fontweight='bold')

bdf = pd.DataFrame({k: pd.Series(v) for k,v in brightness_data.items()}).melt(var_name='Class', value_name='Brightness')
cdf = pd.DataFrame({k: pd.Series(v) for k,v in contrast_data.items()}).melt(var_name='Class', value_name='Contrast')

sns.boxplot(data=bdf, x='Class', y='Brightness', palette=COLORS, ax=axes[0],
            boxprops=dict(alpha=0.7), linewidth=1.2)
sns.stripplot(data=bdf, x='Class', y='Brightness', color='black', alpha=0.2, size=2.5, ax=axes[0])
axes[0].set_title('Brightness (Mean Pixel Intensity)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=20)

sns.boxplot(data=cdf, x='Class', y='Contrast', palette=COLORS, ax=axes[1],
            boxprops=dict(alpha=0.7), linewidth=1.2)
sns.stripplot(data=cdf, x='Class', y='Contrast', color='black', alpha=0.2, size=2.5, ax=axes[1])
axes[1].set_title('Contrast (Pixel Std Dev)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
sp = f'{FIG}/fig4_image_quality.png'
plt.savefig(sp, bbox_inches='tight', dpi=110)
plt.show(); plt.close('all')
print(f'Saved: {sp}')

Saved: /kaggle/working/outputs/figures/fig4_image_quality.png


---
## Section 4 — Stratified 70 / 15 / 15 Split

In [11]:
X = df.index.values
y = df['label'].values

sss1 = StratifiedShuffleSplit(n_splits=1,
    test_size=CFG['val_frac']+CFG['test_frac'], random_state=SEED)
train_idx, temp_idx = next(sss1.split(X, y))

sss2 = StratifiedShuffleSplit(n_splits=1,
    test_size=CFG['test_frac']/(CFG['val_frac']+CFG['test_frac']), random_state=SEED)
val_idx_l, test_idx_l = next(sss2.split(temp_idx, y[temp_idx]))
val_idx  = temp_idx[val_idx_l]
test_idx = temp_idx[test_idx_l]

df_train = df.iloc[train_idx].reset_index(drop=True)
df_val   = df.iloc[val_idx ].reset_index(drop=True)
df_test  = df.iloc[test_idx].reset_index(drop=True)

split_counts = pd.DataFrame({
    'Class': LABELS,
    'Train': [len(df_train[df_train['label']==i]) for i in range(5)],
    'Val'  : [len(df_val  [df_val  ['label']==i]) for i in range(5)],
    'Test' : [len(df_test [df_test ['label']==i]) for i in range(5)],
})
split_counts['Total'] = split_counts[['Train','Val','Test']].sum(axis=1)

print(f'Split summary:  Train={len(df_train)} | Val={len(df_val)} | Test={len(df_test)}')
print()
print(split_counts.to_string(index=False))

df_train.to_csv(f'{TAB}/split_train.csv', index=False)
df_val  .to_csv(f'{TAB}/split_val.csv',   index=False)
df_test .to_csv(f'{TAB}/split_test.csv',  index=False)
split_counts.to_csv(f'{TAB}/split_summary.csv', index=False)

# ── Fig 5: Split distribution
x = np.arange(5); w = 0.26
fig, ax = plt.subplots(figsize=(10, 4))
b1 = ax.bar(x-w, split_counts['Train'], w, label='Train (70%)', color='#3498db', alpha=0.85)
b2 = ax.bar(x,   split_counts['Val'],   w, label='Val   (15%)', color='#2ecc71', alpha=0.85)
b3 = ax.bar(x+w, split_counts['Test'],  w, label='Test  (15%)', color='#e74c3c', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(LABELS)
ax.set_title('Figure 5 — Stratified Train / Val / Test Split per Class', fontweight='bold')
ax.set_ylabel('Image Count'); ax.legend()
for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+3, str(int(h)),
                ha='center', fontsize=7, fontweight='bold')
plt.tight_layout()
sp = f'{FIG}/fig5_split_distribution.png'
plt.savefig(sp, bbox_inches='tight', dpi=110)
plt.show(); plt.close('all')
print(f'Saved: {sp}')

Split summary:  Train=2563 | Val=549 | Test=550

        Class  Train  Val  Test  Total
        No DR   1263  271   271   1805
         Mild    259   55    56    370
     Moderate    699  150   150    999
       Severe    135   29    29    193
Proliferative    207   44    44    295
Saved: /kaggle/working/outputs/figures/fig5_split_distribution.png


---
## Section 5 — Preprocessing: CLAHE + Augmentations

In [12]:
def apply_clahe(img_np):
    """CLAHE on L-channel of LAB colorspace for contrast enhancement."""
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    lab   = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
    lab[:,:,0] = clahe.apply(lab[:,:,0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

# ImageNet stats (pretrained backbone normalisation)
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.3),
    T.RandomRotation(degrees=20),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.RandomAffine(degrees=0, translate=(0.05,0.05), scale=(0.95,1.05)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

val_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

tta_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=10),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

print('✅ Transforms defined.')
print('  Train: H-Flip | V-Flip | Rotation±20° | ColorJitter | Affine | Normalize')
print('  Val  : Normalize only')
print('  TTA  : H-Flip | Rotation±10° | Normalize')

✅ Transforms defined.
  Train: H-Flip | V-Flip | Rotation±20° | ColorJitter | Affine | Normalize
  Val  : Normalize only
  TTA  : H-Flip | Rotation±10° | Normalize


In [13]:
# ════════════════════════════════════════════
# Fig 6: Augmentation Preview
# ════════════════════════════════════════════
sample_path = df_train[df_train['label']==2]['filepath'].iloc[0]
orig        = Image.open(sample_path).convert('RGB').resize((224,224))
orig_np     = np.array(orig)
clahe_np    = apply_clahe(orig_np)
orig_clahe  = Image.fromarray(clahe_np)

aug_list = [
    ('(a) Original',     orig_np),
    ('(b) + CLAHE',      clahe_np),
    ('(c) H-Flip',       np.array(T.RandomHorizontalFlip(p=1)(orig_clahe))),
    ('(d) V-Flip',       np.array(T.RandomVerticalFlip(p=1)(orig_clahe))),
    ('(e) Rotation 20°', np.array(T.RandomRotation(degrees=20)(orig_clahe))),
    ('(f) ColorJitter',  np.array(T.ColorJitter(brightness=0.4, contrast=0.4)(orig_clahe))),
    ('(g) Affine',       np.array(T.RandomAffine(degrees=0, translate=(0.12,0.12))(orig_clahe))),
]
t      = train_transform(orig_clahe)
t_disp = (t * torch.tensor(STD).view(3,1,1) + torch.tensor(MEAN).view(3,1,1))
aug_list.append(('(h) All Combined', (t_disp.permute(1,2,0).clamp(0,1).numpy()*255).astype(np.uint8)))

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
fig.suptitle('Figure 6 — Augmentation Pipeline Preview (Moderate DR Sample)',
             fontsize=12, fontweight='bold')
axes = axes.flatten()
for i,(name,img) in enumerate(aug_list):
    axes[i].imshow(img)
    axes[i].set_title(name, fontsize=9, fontweight='bold')
    axes[i].axis('off')
    for spine in axes[i].spines.values():
        spine.set_visible(True); spine.set_edgecolor('#bdc3c7'); spine.set_linewidth(1)

plt.tight_layout()
sp = f'{FIG}/fig6_augmentation_preview.png'
plt.savefig(sp, bbox_inches='tight', dpi=110)
plt.show(); plt.close('all')
print(f'Saved: {sp}')

Saved: /kaggle/working/outputs/figures/fig6_augmentation_preview.png


---
## Section 6 — Dataset & DataLoaders

In [14]:
class RetinopathyDataset(Dataset):
    """Custom PyTorch dataset for DR retinal images.
    Applies optional CLAHE + configurable transforms.
    """
    def __init__(self, dataframe, transform=None, use_clahe=True):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform
        self.use_clahe = use_clahe

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['filepath'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (CFG['img_size'], CFG['img_size']))
        if self.use_clahe:
            img = apply_clahe(img)
        img = Image.fromarray(img)
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(int(row['label']), dtype=torch.long)


# ── Class weights  (inverse frequency, normalised)
class_counts_arr = np.array([len(df_train[df_train['label']==i]) for i in range(5)])
class_weights_np = 1.0 / class_counts_arr
class_weights_np = class_weights_np / class_weights_np.sum() * 5   # scale to ~1
class_weights    = torch.FloatTensor(class_weights_np).to(CFG['device'])

print('Class weights (inverse-frequency, normalised × 5):')
for i,(l,w) in enumerate(zip(LABELS, class_weights_np)):
    bar = '█' * int(w * 8)
    print(f'  {l:15s} | w={w:.4f} | {bar}')

# ── Datasets
train_ds = RetinopathyDataset(df_train, transform=train_transform, use_clahe=CFG['use_clahe'])
val_ds   = RetinopathyDataset(df_val,   transform=val_transform,   use_clahe=CFG['use_clahe'])
test_ds  = RetinopathyDataset(df_test,  transform=val_transform,   use_clahe=CFG['use_clahe'])

# ── DataLoaders  (num_workers=2 for Kaggle stability)
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'\nDataLoaders ready:')
print(f'  Train batches: {len(train_loader)}  ({len(train_ds)} images)')
print(f'  Val   batches: {len(val_loader)}   ({len(val_ds)} images)')
print(f'  Test  batches: {len(test_loader)}   ({len(test_ds)} images)')

Class weights (inverse-frequency, normalised × 5):
  No DR           | w=0.2161 | █
  Mild            | w=1.0537 | ████████
  Moderate        | w=0.3904 | ███
  Severe          | w=2.0215 | ████████████████
  Proliferative   | w=1.3184 | ██████████

DataLoaders ready:
  Train batches: 80  (2563 images)
  Val   batches: 18   (549 images)
  Test  batches: 18   (550 images)


---
## Section 7 — Model Architectures

In [15]:
# ═══════════════════════════════════════
# Model 1: Baseline CNN (from scratch)
# ═══════════════════════════════════════
class BaselineCNN(nn.Module):
    """
    4-block custom CNN. Used as lower-bound reference.
    Conv→BN→ReLU→MaxPool × 4, then AdaptiveAvgPool + FC head.
    """
    def __init__(self, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,   32,  3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32,  64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64,  128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(256, 256), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

# ═══════════════════════════════════════
# Models 2–4: EfficientNet-B0 (timm)
# ═══════════════════════════════════════
def build_efficientnet_b0(num_classes=5, pretrained=True):
    """EfficientNet-B0 pretrained on ImageNet. Compound-scaled CNN."""
    return timm.create_model('efficientnet_b0', pretrained=pretrained, num_classes=num_classes)

# ═══════════════════════════════════════
# Model 5: ResNet-50 (timm)
# ═══════════════════════════════════════
def build_resnet50(num_classes=5, pretrained=True):
    """ResNet-50 pretrained on ImageNet. Deep residual network."""
    return timm.create_model('resnet50', pretrained=pretrained, num_classes=num_classes)

def count_params(m):
    total     = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

# ── Model summary table
print('┌─────────────────────────┬─────────────────┬─────────────────┐')
print('│ Model                   │ Total Params    │ Trainable Params│')
print('├─────────────────────────┼─────────────────┼─────────────────┤')
# pretrained=False here = NO download. Weights download once at training start.
for name, fn, kw in [
    ('Baseline CNN',    BaselineCNN,           {}),
    ('EfficientNet-B0', build_efficientnet_b0, {'pretrained': False}),
    ('ResNet-50',       build_resnet50,        {'pretrained': False}),
]:
    m = fn(**kw)
    t, tr = count_params(m)
    print(f'│ {name:<23s} │ {t:>14,}  │ {tr:>14,}  │')
    del m
print('└─────────────────────────┴─────────────────┴─────────────────┘')
print()
print('✅ Architecture verified — no pretrained weights downloaded here.')
print('   Pretrained weights download once at training start (timm caches).')


┌─────────────────────────┬─────────────────┬─────────────────┐
│ Model                   │ Total Params    │ Trainable Params│
├─────────────────────────┼─────────────────┼─────────────────┤
│ Baseline CNN            │        456,453  │        456,453  │
│ EfficientNet-B0         │      4,013,953  │      4,013,953  │
│ ResNet-50               │     23,518,277  │     23,518,277  │
└─────────────────────────┴─────────────────┴─────────────────┘

✅ Architecture verified — no pretrained weights downloaded here.
   Pretrained weights download once at training start (timm caches).


---
## Section 8 — Loss Functions

In [16]:
def get_ce_loss():
    """Standard cross-entropy with mild label smoothing."""
    return nn.CrossEntropyLoss(label_smoothing=CFG['label_smooth'])

def get_weighted_ce_loss(weights):
    """Class-weighted CE: upweights minority DR grades."""
    return nn.CrossEntropyLoss(weight=weights, label_smoothing=CFG['label_smooth'])

class FocalLoss(nn.Module):
    """
    Focal Loss (Lin et al. 2017) with alpha scaling.
    FL(p) = -alpha * (1-p)^gamma * log(p)
    gamma=2 focuses training on hard misclassified examples.
    alpha=0.25 rescales to comparable magnitude as CE.
    """
    def __init__(self, gamma=2.0, alpha=0.25, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma          = gamma
        self.alpha          = alpha
        self.weight         = weight
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.weight,
                             label_smoothing=self.label_smoothing, reduction='none')
        pt = torch.exp(-ce)
        return (self.alpha * ((1-pt)**self.gamma) * ce).mean()

def get_focal_loss(weights=None):
    return FocalLoss(
        gamma=CFG['focal_gamma'],
        alpha=CFG['focal_alpha'],
        weight=weights,
        label_smoothing=CFG['label_smooth']
    )

print('Loss functions defined:')
print('  1. CrossEntropyLoss          (label_smooth=0.05)')
print('  2. Weighted CrossEntropyLoss (label_smooth=0.05, class-reweighted)')
print('  3. Focal Loss                (gamma=2.0, alpha=0.25 — magnitude-corrected)')
print()
print('  ✅ FocalLoss fix: alpha=0.25 rescales focal to same magnitude as CE.')
print('     This prevents the near-zero loss values seen in unscaled focal implementations.')

Loss functions defined:
  1. CrossEntropyLoss          (label_smooth=0.05)
  2. Weighted CrossEntropyLoss (label_smooth=0.05, class-reweighted)
  3. Focal Loss                (gamma=2.0, alpha=0.25 — magnitude-corrected)

  ✅ FocalLoss fix: alpha=0.25 rescales focal to same magnitude as CE.
     This prevents the near-zero loss values seen in unscaled focal implementations.


---
## Section 9 — Training Engine

In [17]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device, freeze_backbone=False):
    model.train()
    if freeze_backbone:
        for name, p in model.named_parameters():
            if not any(k in name for k in ['classifier','fc','head']):
                p.requires_grad_(False)

    running_loss, all_preds, all_labels = 0.0, [], []
    for imgs, labels_b in loader:
        imgs, labels_b = imgs.to(device), labels_b.to(device)
        optimizer.zero_grad()
        with amp.autocast(enabled=CFG['use_amp']):
            out  = model(imgs)
            loss = criterion(out, labels_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        running_loss += loss.item() * imgs.size(0)
        all_preds.extend(out.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels_b.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')           if len(set(all_labels)) > 1 else 0.0
    return running_loss / len(loader.dataset), acc, qwk


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, all_preds, all_labels, all_probs = 0.0, [], [], []
    for imgs, labels_b in loader:
        imgs, labels_b = imgs.to(device), labels_b.to(device)
        with amp.autocast(enabled=CFG['use_amp']):
            out  = model(imgs)
            loss = criterion(out, labels_b)
        running_loss += loss.item() * imgs.size(0)
        probs = torch.softmax(out, dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(probs.argmax(axis=1))
        all_labels.extend(labels_b.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')           if len(set(all_labels)) > 1 else 0.0
    return (running_loss/len(loader.dataset), acc, qwk,
            np.array(all_preds), np.array(all_labels), np.array(all_probs))


def train_model(model, train_loader, val_loader, criterion, model_name,
                lr=CFG['lr'], num_epochs=CFG['num_epochs'], patience=CFG['patience']):
    device    = CFG['device']
    model     = model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    scaler    = GradScaler(enabled=CFG['use_amp'])

    history = {k:[] for k in ['train_loss','val_loss','train_acc','val_acc','train_qwk','val_qwk','lr']}
    best_qwk, best_weights, patience_ctr = -1.0, None, 0
    slug      = model_name.replace(' ','_').replace('/','_').replace('(','').replace(')','')
    ckpt_path = f'{CKPT}/{slug}_best.pth'

    print('\n' + '═'*65)
    print(f'  Training: {model_name}')
    print(f'  Optimizer: AdamW | lr={lr} | Scheduler: CosineAnnealingLR')
    print('═'*65)

    is_pretrained = any(k in model_name for k in ['EfficientNet','ResNet'])

    for epoch in range(1, num_epochs+1):
        freeze = is_pretrained and (epoch <= 3)
        if is_pretrained and epoch == 4:
            for p in model.parameters(): p.requires_grad_(True)
            print('  ↳ Backbone unfrozen at epoch 4')

        t0 = time.time()
        tr_loss, tr_acc, tr_qwk = train_one_epoch(
            model, train_loader, optimizer, criterion, scaler, device, freeze)
        vl_loss, vl_acc, vl_qwk, _, _, _ = evaluate(model, val_loader, criterion, device)
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step()

        for k,v in zip(
            ['train_loss','val_loss','train_acc','val_acc','train_qwk','val_qwk','lr'],
            [tr_loss, vl_loss, tr_acc, vl_acc, tr_qwk, vl_qwk, current_lr]
        ):
            history[k].append(v)

        marker = ''
        if vl_qwk > best_qwk:
            best_qwk = vl_qwk
            best_weights = copy.deepcopy(model.state_dict())
            torch.save(best_weights, ckpt_path)
            patience_ctr = 0; marker = ' ✅ BEST'
        else:
            patience_ctr += 1

        print(f'  Ep {epoch:02d}/{num_epochs} | '
              f'Tr[loss={tr_loss:.4f} acc={tr_acc:.3f} qwk={tr_qwk:.3f}] | '
              f'Vl[loss={vl_loss:.4f} acc={vl_acc:.3f} qwk={vl_qwk:.3f}] | '
              f'{time.time()-t0:.1f}s{marker}')

        if patience_ctr >= patience:
            print(f'  ⏹  Early stop — best Val QWK: {best_qwk:.4f}')
            break

    model.load_state_dict(best_weights)
    print(f'  Checkpoint: {ckpt_path}')
    return model, history

print('✅ Training engine ready.')
print('  Features: AMP | CosineAnnealingLR | GradClip | EarlyStopping | Backbone freeze (ep 1-3)')

✅ Training engine ready.
  Features: AMP | CosineAnnealingLR | GradClip | EarlyStopping | Backbone freeze (ep 1-3)


---
## Section 10 — Metrics & Test-Time Augmentation

In [18]:
def compute_metrics(y_true, y_pred, y_probs, model_name):
    acc      = accuracy_score(y_true, y_pred)
    qwk      = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    macro_f1 = f1_score(y_true, y_pred, average='macro',     zero_division=0)
    macro_p  = precision_score(y_true, y_pred, average='macro', zero_division=0)
    macro_r  = recall_score(y_true, y_pred, average='macro',    zero_division=0)
    report   = classification_report(y_true, y_pred,
                   target_names=CFG['class_names'], output_dict=True, zero_division=0)

    # AUROC — referable DR (grade ≥ 2 vs < 2) — primary clinical metric
    ref_true  = (y_true >= 2).astype(int)
    ref_probs = y_probs[:, 2:].sum(axis=1)
    auroc_ref = roc_auc_score(ref_true, ref_probs)

    # Macro AUROC one-vs-rest
    y_bin       = label_binarize(y_true, classes=list(range(5)))
    auroc_macro = roc_auc_score(y_bin, y_probs, average='macro', multi_class='ovr')

    return {
        'Model'            : model_name,
        'Accuracy'         : round(acc,      4),
        'Macro Precision'  : round(macro_p,  4),
        'Macro Recall'     : round(macro_r,  4),
        'Macro F1'         : round(macro_f1, 4),
        'QWK'              : round(qwk,      4),
        'AUROC (Referable)': round(auroc_ref,   4),
        'AUROC (Macro OvR)': round(auroc_macro, 4),
    }, report


@torch.no_grad()
def predict_with_tta(model, dataset, n_tta=CFG['tta_n'], device=CFG['device']):
    """Average softmax probabilities over n_tta augmented passes."""
    model.eval()
    all_probs, all_labels = None, []

    for _, label in DataLoader(dataset, batch_size=1, shuffle=False):
        all_labels.append(label.item())

    for _ in range(n_tta):
        dataset.transform = tta_transform
        loader = DataLoader(dataset, batch_size=CFG['batch_size'],
                            shuffle=False, num_workers=2, pin_memory=True)
        fold = []
        for imgs, _ in loader:
            with amp.autocast(enabled=CFG['use_amp']):
                out = model(imgs.to(device))
            fold.append(torch.softmax(out, dim=1).cpu().numpy())
        fold_arr  = np.concatenate(fold)
        all_probs = fold_arr if all_probs is None else all_probs + fold_arr

    dataset.transform = val_transform  # restore
    all_probs /= n_tta
    return all_probs.argmax(axis=1), np.array(all_labels), all_probs


print(f'✅ Metrics suite: Acc | Precision | Recall | Macro-F1 | QWK | AUROC-Ref | AUROC-Macro')
print(f'✅ TTA: {CFG["tta_n"]} augmented passes averaged at inference')

✅ Metrics suite: Acc | Precision | Recall | Macro-F1 | QWK | AUROC-Ref | AUROC-Macro
✅ TTA: 5 augmented passes averaged at inference


---
## Section 11 — Plotting Helpers

In [19]:
def slug(name):
    return name.replace(' ','_').replace('/','_').replace('(','').replace(')','')

def plot_training_curves(history, model_name):
    e   = range(1, len(history['train_loss'])+1)
    fig = plt.figure(figsize=(17, 4))
    fig.suptitle(f'Training History — {model_name}', fontweight='bold', fontsize=12)
    gs  = gridspec.GridSpec(1, 4, figure=fig)

    pairs = [('Loss','train_loss','val_loss','#e74c3c'),
             ('Accuracy','train_acc','val_acc','#3498db'),
             ('QWK','train_qwk','val_qwk','#2ecc71')]

    for i,(title,trk,vlk,col) in enumerate(pairs):
        ax = fig.add_subplot(gs[i])
        ax.plot(e, history[trk], color=col, lw=2, label='Train')
        ax.plot(e, history[vlk], color=col, lw=2, ls='--', label='Val', alpha=0.7)
        ax.fill_between(e, history[trk], history[vlk], alpha=0.08, color=col)
        best_ep = int(np.argmax(history[vlk]) if 'loss' not in vlk else np.argmin(history[vlk])) + 1
        ax.axvline(best_ep, color='grey', ls=':', lw=1, label=f'Best ep={best_ep}')
        ax.set_title(title, fontweight='bold'); ax.set_xlabel('Epoch')
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # LR schedule
    ax3 = fig.add_subplot(gs[3])
    ax3.plot(e, history['lr'], color='#9b59b6', lw=2)
    ax3.set_title('Learning Rate (CosineAnneal)', fontweight='bold')
    ax3.set_xlabel('Epoch'); ax3.set_ylabel('LR'); ax3.grid(alpha=0.3)

    plt.tight_layout()
    sp = f'{FIG}/curves_{slug(model_name)}.png'
    plt.savefig(sp, bbox_inches='tight', dpi=110)
    plt.show(); plt.close('all')
    print(f'  Saved: {sp}')


def plot_confusion_matrices(y_true, y_pred, model_name):
    cm   = confusion_matrix(y_true, y_pred)
    cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Confusion Matrices — {model_name}', fontweight='bold', fontsize=12)

    for ax, data, fmt, title, cmap in [
        (axes[0], cm,   'd',    'Raw Counts',   'Blues'),
        (axes[1], cm_n, '.2f',  'Row-Normalised','YlOrRd'),
    ]:
        sns.heatmap(data, annot=True, fmt=fmt, cmap=cmap,
                    xticklabels=CFG['class_names'], yticklabels=CFG['class_names'],
                    ax=ax, linewidths=0.5, linecolor='white',
                    cbar_kws={'shrink':0.8})
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Predicted Label', fontweight='bold')
        ax.set_ylabel('True Label', fontweight='bold')
        ax.tick_params(axis='x', rotation=30)

    plt.tight_layout()
    sp = f'{FIG}/cm_{slug(model_name)}.png'
    plt.savefig(sp, bbox_inches='tight', dpi=110)
    plt.show(); plt.close('all')
    print(f'  Saved: {sp}')


def plot_roc_pr(y_true, y_probs, model_name):
    y_bin = label_binarize(y_true, classes=list(range(5)))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'ROC & Precision-Recall Curves — {model_name}', fontweight='bold', fontsize=12)

    for i,(cls_name,c) in enumerate(zip(CFG['class_names'], COLORS)):
        fpr,tpr,_ = roc_curve(y_bin[:,i], y_probs[:,i])
        auc = roc_auc_score(y_bin[:,i], y_probs[:,i])
        axes[0].plot(fpr, tpr, color=c, lw=2, label=f'{cls_name} (AUC={auc:.3f})')

        prec,rec,_ = precision_recall_curve(y_bin[:,i], y_probs[:,i])
        ap = average_precision_score(y_bin[:,i], y_probs[:,i])
        axes[1].plot(rec, prec, color=c, lw=2, label=f'{cls_name} (AP={ap:.3f})')

    axes[0].plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
    axes[0].set_title('ROC Curve (One-vs-Rest)', fontweight='bold')
    axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
    axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
    axes[0].fill_between([0,1],[0,1], alpha=0.04, color='grey')

    axes[1].set_title('Precision-Recall Curve (One-vs-Rest)', fontweight='bold')
    axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
    axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    sp = f'{FIG}/roc_pr_{slug(model_name)}.png'
    plt.savefig(sp, bbox_inches='tight', dpi=110)
    plt.show(); plt.close('all')
    print(f'  Saved: {sp}')


def plot_per_class_metrics(report, model_name):
    rows  = {c: report[c] for c in CFG['class_names']}
    df_m  = pd.DataFrame(rows).T[['precision','recall','f1-score']]
    x     = np.arange(len(CFG['class_names'])); w = 0.25

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(x-w,   df_m['precision'], w, label='Precision', color='#3498db', alpha=0.85)
    ax.bar(x,     df_m['recall'],    w, label='Recall',    color='#e74c3c', alpha=0.85)
    ax.bar(x+w,   df_m['f1-score'],  w, label='F1-Score',  color='#2ecc71', alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(CFG['class_names'], rotation=20, ha='right')
    ax.set_title(f'Per-Class Metrics — {model_name}', fontweight='bold')
    ax.set_ylabel('Score'); ax.set_ylim(0, 1.09); ax.legend(); ax.grid(axis='y', alpha=0.3)
    for bars in ax.containers:
        ax.bar_label(bars, fmt='%.2f', fontsize=7, padding=2)

    plt.tight_layout()
    sp = f'{FIG}/perclass_{slug(model_name)}.png'
    plt.savefig(sp, bbox_inches='tight', dpi=110)
    plt.show(); plt.close('all')
    print(f'  Saved: {sp}')


print('✅ All plotting helpers ready.')

✅ All plotting helpers ready.


---
## Section 12 — Grad-CAM++ Explainability

In [20]:
def get_cam_target_layer(model, model_name):
    if 'EfficientNet' in model_name:
        return [model.conv_head]
    elif 'ResNet' in model_name:
        return [model.layer4[-1]]
    else:
        return [model.features[-3]]   # last Conv2d in BaselineCNN


def run_gradcam(model, model_name, df_subset, n_samples=8):
    """Grad-CAM++ visualisation: original vs activation map side-by-side."""
    model.eval()
    device = CFG['device']

    try:
        cam = GradCAMPlusPlus(model=model, target_layers=get_cam_target_layer(model, model_name))
    except Exception as e:
        print(f'  ⚠️  Grad-CAM init failed: {e}'); return

    # 2 samples per class
    samples_list = []
    for cls in range(5):
        sub = df_subset[df_subset['label']==cls]
        if len(sub) > 0:
            samples_list.append(sub.sample(min(2,len(sub)), random_state=SEED))
    samples = pd.concat(samples_list).head(n_samples).reset_index(drop=True)

    n_rows = (len(samples)+1)//2
    fig, axes = plt.subplots(n_rows, 4, figsize=(15, n_rows*3.8))
    axes = np.array(axes).reshape(n_rows, 4)
    fig.suptitle(f'Grad-CAM++ Activation Maps — {model_name}',
                 fontweight='bold', fontsize=12)

    for i, (_, row) in enumerate(samples.iterrows()):
        img_np = cv2.cvtColor(cv2.imread(row['filepath']), cv2.COLOR_BGR2RGB)
        img_np = cv2.resize(img_np, (224,224))
        if CFG['use_clahe']:
            img_np = apply_clahe(img_np)

        tensor = val_transform(Image.fromarray(img_np)).unsqueeze(0).to(device)

        try:
            gray_cam = cam(input_tensor=tensor,
                           targets=[ClassifierOutputTarget(int(row['label']))])[0]
            cam_img  = show_cam_on_image(img_np.astype(np.float32)/255., gray_cam, use_rgb=True)
        except Exception as e:
            print(f'  ⚠️  Sample {i} failed: {e}'); continue

        with torch.no_grad():
            pred = model(tensor).argmax(dim=1).item()

        r, c = i//2, (i%2)*2
        true_cls = int(row['label'])
        correct  = pred == true_cls

        axes[r][c].imshow(img_np)
        axes[r][c].set_title(f'True: {CFG["class_names"][true_cls]}',
                              fontsize=8, fontweight='bold', color=COLORS[true_cls])
        axes[r][c].axis('off')

        border_col = '#27ae60' if correct else '#e74c3c'
        for spine in axes[r][c+1].spines.values():
            spine.set_visible(True); spine.set_edgecolor(border_col); spine.set_linewidth(2.5)
        axes[r][c+1].imshow(cam_img)
        tick = '✓' if correct else '✗'
        axes[r][c+1].set_title(f'Pred: {CFG["class_names"][pred]} {tick}',
                                fontsize=8, fontweight='bold', color=border_col)
        axes[r][c+1].axis('off')

    # Hide unused
    for j in range(len(samples), n_rows*2):
        r,c = j//2, (j%2)*2
        if r < n_rows:
            axes[r][c].axis('off'); axes[r][c+1].axis('off')

    plt.tight_layout()
    sp = f'{GCAM}/gradcam_{slug(model_name)}.png'
    plt.savefig(sp, bbox_inches='tight', dpi=110)
    plt.show(); plt.close('all')
    print(f'  Saved: {sp}')


print('✅ Grad-CAM++ ready.')

✅ Grad-CAM++ ready.


---
## Section 13 — Experiment Runner

Five models trained sequentially. Results stored in `all_results` dict.

| # | Model | Loss | Purpose |
|---|---|---|---|
| 1 | Baseline CNN | Weighted CE | Lower-bound reference |
| 2 | EfficientNet-B0 | CE | Pretrained baseline |
| 3 | EfficientNet-B0 | Weighted CE | **Primary model** |
| 4 | EfficientNet-B0 | Focal Loss | Minority class focus |
| 5 | ResNet-50 | Weighted CE | Architecture comparison |

In [21]:
experiments = [
    {
        'name'    : 'Baseline CNN (Weighted CE)',
        'model_fn': lambda: BaselineCNN(num_classes=5),
        'loss_fn' : lambda: get_weighted_ce_loss(class_weights),
    },
    {
        'name'    : 'EfficientNet-B0 (CE)',
        'model_fn': lambda: build_efficientnet_b0(),
        'loss_fn' : lambda: get_ce_loss(),
    },
    {
        'name'    : 'EfficientNet-B0 (Weighted CE)',
        'model_fn': lambda: build_efficientnet_b0(),
        'loss_fn' : lambda: get_weighted_ce_loss(class_weights),
    },
    {
        'name'    : 'EfficientNet-B0 (Focal Loss)',
        'model_fn': lambda: build_efficientnet_b0(),
        'loss_fn' : lambda: get_focal_loss(class_weights),
    },
    {
        'name'    : 'ResNet-50 (Weighted CE)',
        'model_fn': lambda: build_resnet50(),
        'loss_fn' : lambda: get_weighted_ce_loss(class_weights),
    },
]

print(f'{len(experiments)} experiments queued:')
for i,e in enumerate(experiments, 1):
    print(f'  {i}. {e["name"]}')

5 experiments queued:
  1. Baseline CNN (Weighted CE)
  2. EfficientNet-B0 (CE)
  3. EfficientNet-B0 (Weighted CE)
  4. EfficientNet-B0 (Focal Loss)
  5. ResNet-50 (Weighted CE)


In [22]:
# ══════════════════════════════════════════════════════════
#  MAIN TRAINING LOOP
# ══════════════════════════════════════════════════════════
set_seed()
all_results = {}

for exp in experiments:
    name = exp['name']
    print(f'\n🚀  {name}')

    model     = exp['model_fn']()
    criterion = exp['loss_fn']()

    model, history = train_model(model, train_loader, val_loader, criterion, name)

    # TTA evaluation
    print(f'  → TTA evaluation (n={CFG["tta_n"]})...')
    y_pred, y_true, y_probs = predict_with_tta(model, test_ds)
    metrics, report = compute_metrics(y_true, y_pred, y_probs, name)

    all_results[name] = {
        'history': history, 'metrics': metrics, 'report': report,
        'y_true': y_true, 'y_pred': y_pred, 'y_probs': y_probs,
    }

    m = metrics
    print(f'  ✅ Acc={m["Accuracy"]} QWK={m["QWK"]} F1={m["Macro F1"]} AUROC={m["AUROC (Referable)"]}')

    # Per-model plots
    plot_training_curves(history, name)
    plot_confusion_matrices(y_true, y_pred, name)
    plot_roc_pr(y_true, y_probs, name)
    plot_per_class_metrics(report, name)

    if 'EfficientNet' in name or 'ResNet' in name:
        run_gradcam(model, name, df_test, n_samples=8)

    model.cpu(); del model
    torch.cuda.empty_cache()

print('\n🎉  All 5 experiments complete!')


🚀  Baseline CNN (Weighted CE)

═════════════════════════════════════════════════════════════════
  Training: Baseline CNN (Weighted CE)
  Optimizer: AdamW | lr=0.0001 | Scheduler: CosineAnnealingLR
═════════════════════════════════════════════════════════════════
  Ep 01/20 | Tr[loss=1.6218 acc=0.405 qwk=0.265] | Vl[loss=1.5216 acc=0.648 qwk=0.480] | 22.6s ✅ BEST
  Ep 02/20 | Tr[loss=1.4958 acc=0.539 qwk=0.474] | Vl[loss=1.5262 acc=0.355 qwk=0.155] | 13.7s
  Ep 03/20 | Tr[loss=1.4589 acc=0.537 qwk=0.506] | Vl[loss=1.4250 acc=0.536 qwk=0.411] | 13.8s
  Ep 04/20 | Tr[loss=1.4280 acc=0.565 qwk=0.536] | Vl[loss=1.4637 acc=0.464 qwk=0.300] | 13.4s
  Ep 05/20 | Tr[loss=1.4143 acc=0.568 qwk=0.542] | Vl[loss=1.4231 acc=0.587 qwk=0.532] | 13.8s ✅ BEST
  Ep 06/20 | Tr[loss=1.4105 acc=0.572 qwk=0.564] | Vl[loss=1.4170 acc=0.541 qwk=0.413] | 13.8s
  Ep 07/20 | Tr[loss=1.3928 acc=0.598 qwk=0.592] | Vl[loss=1.4069 acc=0.546 qwk=0.405] | 13.8s
  Ep 08/20 | Tr[loss=1.3803 acc=0.606 qwk=0.610] | Vl[lo

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]


═════════════════════════════════════════════════════════════════
  Training: EfficientNet-B0 (CE)
  Optimizer: AdamW | lr=0.0001 | Scheduler: CosineAnnealingLR
═════════════════════════════════════════════════════════════════
  Ep 01/20 | Tr[loss=2.0057 acc=0.505 qwk=0.381] | Vl[loss=1.8090 acc=0.559 qwk=0.490] | 20.8s ✅ BEST
  Ep 02/20 | Tr[loss=1.4193 acc=0.632 qwk=0.591] | Vl[loss=1.5937 acc=0.596 qwk=0.582] | 13.5s ✅ BEST
  Ep 03/20 | Tr[loss=1.3003 acc=0.664 qwk=0.629] | Vl[loss=1.4631 acc=0.603 qwk=0.627] | 13.8s ✅ BEST
  ↳ Backbone unfrozen at epoch 4
  Ep 04/20 | Tr[loss=1.1702 acc=0.684 qwk=0.691] | Vl[loss=1.2895 acc=0.676 qwk=0.692] | 20.3s ✅ BEST
  Ep 05/20 | Tr[loss=0.9616 acc=0.732 qwk=0.760] | Vl[loss=1.2392 acc=0.716 qwk=0.684] | 14.3s
  Ep 06/20 | Tr[loss=0.8870 acc=0.753 qwk=0.779] | Vl[loss=1.2208 acc=0.703 qwk=0.645] | 14.1s
  Ep 07/20 | Tr[loss=0.8168 acc=0.765 qwk=0.783] | Vl[loss=1.1249 acc=0.725 qwk=0.695] | 14.3s ✅ BEST
  Ep 08/20 | Tr[loss=0.7364 acc=0.802 q

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]


═════════════════════════════════════════════════════════════════
  Training: ResNet-50 (Weighted CE)
  Optimizer: AdamW | lr=0.0001 | Scheduler: CosineAnnealingLR
═════════════════════════════════════════════════════════════════
  Ep 01/20 | Tr[loss=1.6659 acc=0.121 qwk=0.025] | Vl[loss=1.6501 acc=0.242 qwk=0.183] | 14.6s ✅ BEST
  Ep 02/20 | Tr[loss=1.6328 acc=0.431 qwk=0.365] | Vl[loss=1.6249 acc=0.446 qwk=0.444] | 14.1s ✅ BEST
  Ep 03/20 | Tr[loss=1.6046 acc=0.516 qwk=0.457] | Vl[loss=1.5977 acc=0.494 qwk=0.484] | 14.0s ✅ BEST
  ↳ Backbone unfrozen at epoch 4
  Ep 04/20 | Tr[loss=1.5148 acc=0.577 qwk=0.524] | Vl[loss=1.4463 acc=0.550 qwk=0.518] | 15.3s ✅ BEST
  Ep 05/20 | Tr[loss=1.3955 acc=0.598 qwk=0.596] | Vl[loss=1.3808 acc=0.630 qwk=0.689] | 15.1s ✅ BEST
  Ep 06/20 | Tr[loss=1.3319 acc=0.626 qwk=0.670] | Vl[loss=1.3391 acc=0.614 qwk=0.730] | 15.0s ✅ BEST
  Ep 07/20 | Tr[loss=1.2705 acc=0.655 qwk=0.734] | Vl[loss=1.2962 acc=0.627 qwk=0.750] | 15.0s ✅ BEST
  Ep 08/20 | Tr[loss=1

---
## Section 14 — Final Comparison

In [23]:
# ── Metrics table
metrics_list = [v['metrics'] for v in all_results.values()]
df_metrics   = pd.DataFrame(metrics_list).sort_values('QWK', ascending=False).reset_index(drop=True)
df_metrics.index = df_metrics.index + 1   # rank from 1

print('━'*95)
print('  FINAL RESULTS TABLE  (sorted by QWK — primary metric)')
print('━'*95)
display_cols = ['Model','Accuracy','Macro Precision','Macro Recall','Macro F1','QWK','AUROC (Referable)','AUROC (Macro OvR)']
print(df_metrics[display_cols].to_string())
print('━'*95)
print(f'  🏆 Best model: {df_metrics.iloc[0]["Model"]}')
print(f'     QWK={df_metrics.iloc[0]["QWK"]} | Acc={df_metrics.iloc[0]["Accuracy"]} | AUROC={df_metrics.iloc[0]["AUROC (Referable)"]}')

df_metrics.to_csv(f'{TAB}/final_metrics.csv', index=True)
print(f'\n  Saved: {TAB}/final_metrics.csv')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  FINAL RESULTS TABLE  (sorted by QWK — primary metric)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                           Model  Accuracy  Macro Precision  Macro Recall  Macro F1     QWK  AUROC (Referable)  AUROC (Macro OvR)
1           EfficientNet-B0 (CE)    0.8200           0.6887        0.6305    0.6528  0.8582             0.9786             0.9245
2   EfficientNet-B0 (Focal Loss)    0.7364           0.5985        0.6522    0.5923  0.8282             0.9752             0.9094
3        ResNet-50 (Weighted CE)    0.6927           0.5266        0.5618    0.5050  0.8006             0.9763             0.9072
4  EfficientNet-B0 (Weighted CE)    0.6436           0.5315        0.5783    0.4955  0.7830             0.9583             0.8724
5     Baseline CNN (Weighted CE)    0.6727           0.5163        0.4970    0.4778  0.7182           

In [24]:
# ════════════════════════════════════════════
# Fig 7: Comprehensive comparison — 4 metrics
# ════════════════════════════════════════════
short_names = [
    m.replace('EfficientNet-B0','EffB0').replace('ResNet-50','R50').replace('Baseline CNN','CNN')
    for m in df_metrics['Model']
]
metrics_to_plot = ['Accuracy','Macro F1','QWK','AUROC (Referable)']
palette = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#27ae60'][:len(df_metrics)]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Figure 7 — Final Model Comparison Across Key Metrics', fontsize=13, fontweight='bold')

for ax, metric in zip(axes, metrics_to_plot):
    vals  = df_metrics[metric].values
    bars  = ax.bar(range(len(df_metrics)), vals, color=palette, edgecolor='white', linewidth=1.2, width=0.6)
    ax.set_xticks(range(len(df_metrics)))
    ax.set_xticklabels(short_names, rotation=35, ha='right', fontsize=8)
    ax.set_title(metric, fontweight='bold')
    ax.set_ylim(max(0, vals.min()-0.06), min(1.0, vals.max()+0.07))
    ax.grid(axis='y', alpha=0.3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.004,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    # Highlight best
    best_i = int(np.argmax(vals))
    bars[best_i].set_edgecolor('#2c3e50'); bars[best_i].set_linewidth(2.5)

plt.tight_layout()
sp = f'{FIG}/fig7_final_comparison.png'
plt.savefig(sp, bbox_inches='tight', dpi=120)
plt.show(); plt.close('all')
print(f'Saved: {sp}')

Saved: /kaggle/working/outputs/figures/fig7_final_comparison.png


In [25]:
# ════════════════════════════════════════════
# Fig 8: Radar chart — all 5 models on 4 metrics
# ════════════════════════════════════════════
radar_metrics = ['Accuracy','Macro F1','QWK','AUROC (Referable)']
N = len(radar_metrics)
angles = [n/float(N)*2*np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8,8), subplot_kw=dict(polar=True))
fig.suptitle('Figure 8 — Radar Chart: Multi-Metric Model Comparison', fontweight='bold', fontsize=12)

model_colors = ['#95a5a6','#3498db','#e74c3c','#e67e22','#9b59b6']
for i, (_, row) in enumerate(df_metrics.iterrows()):
    vals   = [row[m] for m in radar_metrics]
    vals  += vals[:1]
    name_s = row['Model'].replace('EfficientNet-B0','EffB0').replace('ResNet-50','R50').replace('Baseline CNN','CNN')
    ax.plot(angles, vals, color=model_colors[i], lw=2, label=name_s)
    ax.fill(angles, vals, color=model_colors[i], alpha=0.06)

ax.set_xticks(angles[:-1]); ax.set_xticklabels(radar_metrics, fontsize=10, fontweight='bold')
ax.set_ylim(0, 1); ax.set_yticks([0.2,0.4,0.6,0.8,1.0])
ax.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], fontsize=7)
ax.legend(loc='upper right', bbox_to_anchor=(1.35,1.15), fontsize=9)
ax.grid(color='grey', alpha=0.3)

plt.tight_layout()
sp = f'{FIG}/fig8_radar_comparison.png'
plt.savefig(sp, bbox_inches='tight', dpi=120)
plt.show(); plt.close('all')
print(f'Saved: {sp}')

Saved: /kaggle/working/outputs/figures/fig8_radar_comparison.png


In [26]:
# ════════════════════════════════════════════
# Fig 9: Per-class recall heatmap across all 5 models
# ════════════════════════════════════════════
recall_matrix = []
for name in df_metrics['Model']:
    res = all_results[name]
    cm  = confusion_matrix(res['y_true'], res['y_pred'])
    per_cls = cm.diagonal() / cm.sum(axis=1)
    recall_matrix.append(per_cls)

recall_df = pd.DataFrame(
    recall_matrix,
    index=[m.replace('EfficientNet-B0','EffB0').replace('ResNet-50','R50').replace('Baseline CNN','CNN')
           for m in df_metrics['Model']],
    columns=CFG['class_names']
)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(recall_df, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, linecolor='white', vmin=0, vmax=1,
            cbar_kws={'label':'Recall','shrink':0.8}, ax=ax)
ax.set_title('Figure 9 — Per-Class Recall Heatmap Across All Models', fontweight='bold', fontsize=12)
ax.set_xlabel('DR Severity Class', fontweight='bold')
ax.set_ylabel('Model', fontweight='bold')
ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
sp = f'{FIG}/fig9_recall_heatmap.png'
plt.savefig(sp, bbox_inches='tight', dpi=120)
plt.show(); plt.close('all')
print(f'Saved: {sp}')

Saved: /kaggle/working/outputs/figures/fig9_recall_heatmap.png


---
## Section 15 — Clinical Threshold Optimization

In [27]:
# ── Optimize threshold for referable DR (grade ≥ 2)
best_model_name = df_metrics.iloc[0]['Model']
best_res        = all_results[best_model_name]

y_true_ref = (best_res['y_true'] >= 2).astype(int)
ref_probs  = best_res['y_probs'][:, 2:].sum(axis=1)

thresholds = np.linspace(0.05, 0.95, 181)
sensitivities, specificities, f1s, ppvs = [], [], [], []

for t in thresholds:
    preds = (ref_probs >= t).astype(int)
    tp = ((preds==1)&(y_true_ref==1)).sum()
    fp = ((preds==1)&(y_true_ref==0)).sum()
    tn = ((preds==0)&(y_true_ref==0)).sum()
    fn = ((preds==0)&(y_true_ref==1)).sum()
    sensitivities.append(tp/(tp+fn) if (tp+fn)>0 else 0)
    specificities.append(tn/(tn+fp) if (tn+fp)>0 else 0)
    f1s.append(f1_score(y_true_ref, preds, zero_division=0))
    ppvs.append(tp/(tp+fp) if (tp+fp)>0 else 0)

best_i      = int(np.argmax(f1s))
best_thresh = thresholds[best_i]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'Figure 10 — Clinical Threshold Optimization ({best_model_name})',
             fontweight='bold', fontsize=12)

axes[0].plot(thresholds, sensitivities, color='#e74c3c', lw=2, label='Sensitivity (Recall)')
axes[0].plot(thresholds, specificities, color='#3498db', lw=2, label='Specificity')
axes[0].axvline(best_thresh, color='black', ls='--', lw=1.5, label=f'Opt. t={best_thresh:.2f}')
axes[0].set_xlabel('Threshold'); axes[0].set_ylabel('Score')
axes[0].set_title('Sensitivity vs Specificity', fontweight='bold')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

axes[1].plot(thresholds, f1s, color='#2ecc71', lw=2, label='F1 Score')
axes[1].plot(thresholds, ppvs, color='#9b59b6', lw=2, label='Precision (PPV)')
axes[1].axvline(best_thresh, color='black', ls='--', lw=1.5, label=f'Opt. t={best_thresh:.2f}')
axes[1].set_xlabel('Threshold'); axes[1].set_ylabel('Score')
axes[1].set_title('F1 & Precision vs Threshold', fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

# Operating point scatter
ax2 = axes[2]
sc  = ax2.scatter(1-np.array(specificities), sensitivities,
                  c=thresholds, cmap='plasma', s=20, alpha=0.7)
ax2.scatter(1-specificities[best_i], sensitivities[best_i],
            color='red', s=150, marker='*', zorder=5, label=f'Optimal t={best_thresh:.2f}')
ax2.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
ax2.set_xlabel('1 - Specificity (FPR)'); ax2.set_ylabel('Sensitivity (TPR)')
ax2.set_title('ROC Operating Point Selection', fontweight='bold')
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)
plt.colorbar(sc, ax=ax2, label='Threshold')

plt.tight_layout()
sp = f'{FIG}/fig10_threshold_optimization.png'
plt.savefig(sp, bbox_inches='tight', dpi=120)
plt.show(); plt.close('all')
print(f'Saved: {sp}')

thresh_results = {
    'Optimal Threshold': round(best_thresh, 3),
    'Sensitivity'      : round(sensitivities[best_i], 3),
    'Specificity'      : round(specificities[best_i], 3),
    'F1 Score'         : round(f1s[best_i], 3),
    'PPV (Precision)'  : round(ppvs[best_i], 3),
}
print('\n  Clinical operating point at optimal threshold:')
for k,v in thresh_results.items():
    print(f'    {k:<22s}: {v}')
pd.DataFrame([thresh_results]).to_csv(f'{TAB}/threshold_results.csv', index=False)

Saved: /kaggle/working/outputs/figures/fig10_threshold_optimization.png

  Clinical operating point at optimal threshold:
    Optimal Threshold     : 0.435
    Sensitivity           : 0.933
    Specificity           : 0.933
    F1 Score              : 0.918
    PPV (Precision)       : 0.904


---
## Section 16 — Key Findings & Interpretation

In [28]:
print('━'*70)
print('  KEY FINDINGS & MODEL INTERPRETATION')
print('━'*70)

interpretation_rows = []

for name, res in all_results.items():
    m  = res['metrics']
    cm = confusion_matrix(res['y_true'], res['y_pred'])
    per_cls_recall = cm.diagonal() / cm.sum(axis=1)
    worst_cls = per_cls_recall.argmin()
    best_cls  = per_cls_recall.argmax()

    print(f'\n🔹 {name}')
    print(f'   QWK={m["QWK"]} | Acc={m["Accuracy"]} | F1={m["Macro F1"]} | AUROC_ref={m["AUROC (Referable)"]}')
    print(f'   Per-class recall: ' + ' | '.join([f'{LABELS[i]}={per_cls_recall[i]:.2f}' for i in range(5)]))
    print(f'   Best recall  → {LABELS[best_cls]}  ({per_cls_recall[best_cls]:.2f})')
    print(f'   Worst recall → {LABELS[worst_cls]} ({per_cls_recall[worst_cls]:.2f})')

    notes = []
    if 'Baseline' in name:
        notes.append('Lower-bound reference. Scratch CNN limited by capacity — cannot capture fine retinal features.')
    if 'CE)' in name and 'Weighted' not in name and 'Focal' not in name:
        notes.append('Unweighted CE: model biases toward majority No DR class. High accuracy hides weak minority recall.')
    if 'Weighted CE' in name and 'EfficientNet' in name:
        notes.append('Primary model. Class weighting penalises Severe/Proliferative misclassifications more heavily.')
    if 'Focal' in name:
        notes.append('Focal loss (γ=2, α=0.25): emphasises hard examples. Alpha scaling corrects prior magnitude issue.')
    if 'ResNet' in name:
        notes.append('ResNet-50: 5× more parameters than EffB0. Competitive QWK with lower accuracy — may benefit from longer training.')

    auroc = m['AUROC (Referable)']
    if   auroc >= 0.95: notes.append(f'AUROC={auroc} — excellent clinical screening performance.')
    elif auroc >= 0.90: notes.append(f'AUROC={auroc} — strong for clinical screening use.')
    elif auroc >= 0.80: notes.append(f'AUROC={auroc} — acceptable for screening; improvement possible.')
    else:               notes.append(f'AUROC={auroc} — below clinical threshold.')

    for note in notes:
        print(f'   → {note}')

    interpretation_rows.append({
        'Model'       : name,
        'QWK'         : m['QWK'],
        'Accuracy'    : m['Accuracy'],
        'Macro F1'    : m['Macro F1'],
        'AUROC Ref'   : m['AUROC (Referable)'],
        'Best Recall Class'  : LABELS[best_cls],
        'Worst Recall Class' : LABELS[worst_cls],
        'Notes'       : ' '.join(notes),
    })

best_name = df_metrics.iloc[0]['Model']
best_row  = df_metrics.iloc[0]
print('\n' + '━'*70)
print(f'  🏆 BEST MODEL: {best_name}')
print(f'     QWK={best_row["QWK"]} | Accuracy={best_row["Accuracy"]} | AUROC={best_row["AUROC (Referable)"]}')
print('━'*70)
print()
print('  KEY TAKEAWAYS:')
print('  1. EfficientNet-B0 (CE) achieves highest QWK — pretrained features outperform from-scratch CNN.')
print('  2. Class imbalance (9.4×) is the primary challenge — Severe class has fewest samples.')
print('  3. Weighted CE improves minority recall at cost of slight accuracy reduction vs plain CE.')
print('  4. AUROC ≥ 0.94 for top models — clinically viable for automated DR screening.')
print('  5. Grad-CAM++ confirms models attend to optic disc & lesion regions — not background artefacts.')

pd.DataFrame(interpretation_rows).to_csv(f'{TAB}/interpretation_table.csv', index=False)
print(f'\n  Saved: {TAB}/interpretation_table.csv')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  KEY FINDINGS & MODEL INTERPRETATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 Baseline CNN (Weighted CE)
   QWK=0.7182 | Acc=0.6727 | F1=0.4778 | AUROC_ref=0.9385
   Per-class recall: No DR=0.97 | Mild=0.38 | Moderate=0.41 | Severe=0.48 | Proliferative=0.25
   Best recall  → No DR  (0.97)
   Worst recall → Proliferative (0.25)
   → Lower-bound reference. Scratch CNN limited by capacity — cannot capture fine retinal features.
   → AUROC=0.9385 — strong for clinical screening use.

🔹 EfficientNet-B0 (CE)
   QWK=0.8582 | Acc=0.82 | F1=0.6528 | AUROC_ref=0.9786
   Per-class recall: No DR=0.99 | Mild=0.50 | Moderate=0.83 | Severe=0.38 | Proliferative=0.45
   Best recall  → No DR  (0.99)
   Worst recall → Severe (0.38)
   → Unweighted CE: model biases toward majority No DR class. High accuracy hides weak minority recall.
   → AUROC=0.9786 — excellent clinical screening performance.

🔹 Eff

In [47]:
import shutil

shutil.make_archive(
    '/kaggle/working/retinaguard_outputs',  # zip name
    'zip',
    '/kaggle/working/outputs'               # folder to zip
)

print("✅ Created: /kaggle/working/retinaguard_outputs.zip")

✅ Created: /kaggle/working/retinaguard_outputs.zip
